# Speech Segmentation using Histogram Algorithm (Giannakopoulos)

This notebook covers the complete workflow for speech/silence segmentation.

**Workflow:**
1. **Training Phase:** Load training signals, extract features (STE & Spectral Centroid), build histograms, compute local maxima (M1, M2), and determine the optimal global thresholds (T1, T2) by searching for the minimum Mean Absolute Error (MAE).
2. **Testing Phase:** Apply the global thresholds to the test signals to predict speech boundaries, evaluate accuracy against ground truth, and display comparative graphs.

In [19]:
import glob
import json
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
import scipy.io.wavfile as wavfile

## 1. Parameters & Configuration
Set up constants, algorithm parameters, and grid search values.

In [20]:
# Fixed parameters
FRAME_MS = 25          # Frame duration (ms)
SHIFT_MS = 10          # Frame shift/step (ms)
MIN_SILENCE_MS = 200   # Minimum silence duration to preserve (ms)

# Algorithm parameters
HIST_BINS = 100        # Number of histogram bins
HIST_SMOOTH_LEN = 5    # Length of histogram smoothing filter
TIME_SMOOTH_LEN = 5    # Length of time-domain feature smoothing filter
PEAK_RATIO = 0.05      # Acceptable peak ratio

# Grid Search arrays for optimization
W_GRID = [0.5, 1, 2, 3, 5, 8, 10, 15, 20]     # Weight for threshold calculation
T_HIGH_GRID = [0.05, 0.1, 0.2, 0.4, 1.0]      # High threshold for STE
H_GRID = [0, 1, 2, 3, 5]                      # Hangover frames (both sides)
SHRINK_GRID = [-3, -2, -1, 0, 1, 2, 3, 4]     # Frames to shrink boundaries
SPEECH_LABELS = {'v', 'uv'}                   # Ground truth labels for speech

# Directories & Files
TRAIN_DIR = "TinHieuHuanLuyen"
TEST_DIR = "TinHieuKiemThu"
THRESHOLD_FILE = "thresholds.json"

## 2. Core Functions
Functions for file reading, feature extraction, histogram analysis, threshold calculation, boundary extraction, and error calculation.

In [21]:
# Read WAV file and normalize
def read_wav(path):
    with warnings.catch_warnings():          
        warnings.simplefilter('ignore')
        fs, x = wavfile.read(path)
    if x.ndim > 1:                       
        x = x.mean(axis=1)
    if np.issubdtype(x.dtype, np.integer):
        x = x / float(np.iinfo(x.dtype).max + 1)
    return x.astype(np.float64), fs

# Read ground truth labels (.lab files)
def read_lab(lab_path, speech_labels=SPEECH_LABELS):
    segs = []
    if not os.path.exists(lab_path):
        return segs
    with open(lab_path, 'r') as f:
        for line in f:
            p = line.strip().split()
            if len(p) >= 3 and p[2].lower() in speech_labels:
                s, e = float(p[0]), float(p[1])
                if segs and abs(segs[-1][1] - s) < 1e-9:
                    segs[-1][1] = e      
                else:
                    segs.append([s, e])
    return segs

# Create overlapping frames
def create_frames(signal, fs, frame_ms=FRAME_MS, shift_ms=SHIFT_MS):
    L = int(round(frame_ms * fs / 1000))
    S = int(round(shift_ms * fs / 1000))
    n = (len(signal) - L) // S + 1
    idx = np.arange(L)[None, :] + S * np.arange(n)[:, None]
    return signal[idx]

# Apply moving average filter
def moving_average(x, length):
    x = np.asarray(x, dtype=np.float64)
    h = length // 2
    padded = np.concatenate([np.full(h, x[0]), x, np.full(h, x[-1])])
    csum = np.concatenate([[0.0], np.cumsum(padded)])
    return (csum[length:] - csum[:-length]) / length

# Compute Short-Time Energy (STE)
def compute_ste(frames):
    ste = np.mean(frames ** 2, axis=1)
    m = np.max(ste)
    return ste / m if m > 0 else ste

# Compute Discrete Fourier Transform Magnitude
def dft_magnitude_half(frames):
    N = frames.shape[1]
    n = np.arange(N)[:, None]
    k = np.arange(N // 2)[None, :]
    ang = 2 * np.pi * n * k / N
    re = frames @ np.cos(ang)
    im = frames @ np.sin(ang)
    return np.sqrt(re ** 2 + im ** 2)

# Compute Spectral Centroid (SC)
def compute_spectral_centroid(frames):
    X = dft_magnitude_half(frames)
    k = np.arange(1, X.shape[1] + 1)[None, :]
    s = np.sum(X, axis=1)
    sc = np.where(s > 0, np.sum(k * X, axis=1) / np.where(s > 0, s, 1), 0.0)
    m = np.max(sc)
    return sc / m if m > 0 else sc

# Extract combined smoothed features
def extract_smoothed_features(path):
    signal, fs = read_wav(path)
    frames = create_frames(signal, fs)
    ste = compute_ste(frames)
    sc = compute_spectral_centroid(frames)
    return {"signal": signal, "fs": fs, "ste": ste, "sc": sc,
            "ste_s": moving_average(ste, TIME_SMOOTH_LEN),
            "sc_s": moving_average(sc, TIME_SMOOTH_LEN)}

# Build histogram for feature data
def build_histogram(features, bins=HIST_BINS, lo=None, hi=None):
    features = np.asarray(features)
    lo = features.min() if lo is None else lo
    hi = features.max() if hi is None else hi
    idx = np.clip(((features - lo) / (hi - lo) * bins).astype(int), 0, bins - 1)
    hist = np.zeros(bins)
    for k in idx:
        hist[k] += 1
    edges = np.linspace(lo, hi, bins + 1)
    return hist, (edges[:-1] + edges[1:]) / 2

# Identify first two local maxima (M1, M2) from histogram
def find_first_two_peaks(smooth_hist, centers, peak_ratio=PEAK_RATIO):
    min_h = peak_ratio * smooth_hist.max()
    peaks = []
    for i in range(len(smooth_hist)):
        left = smooth_hist[i - 1] if i > 0 else -np.inf
        right = smooth_hist[i + 1] if i < len(smooth_hist) - 1 else -np.inf
        if smooth_hist[i] > left and smooth_hist[i] >= right and smooth_hist[i] >= min_h:
            peaks.append(centers[i])
    M1 = peaks[0] if len(peaks) >= 1 else centers[0]
    M2 = peaks[1] if len(peaks) >= 2 else M1
    return M1, M2

# Calculate threshold using M1, M2, and weight W
def threshold_from_peaks(M1, M2, W):
    return (W * M1 + M2) / (W + 1)

# Full histogram analysis pipeline
def analyze_histogram(features):
    hist, centers = build_histogram(features)
    smooth = moving_average(hist, HIST_SMOOTH_LEN)
    M1, M2 = find_first_two_peaks(smooth, centers)
    return M1, M2, hist, smooth, centers

# Classify frame: Speech vs Silence
def classify_frames(ste_s, sc_s, T1, T2, T1_high=1.0):
    return ((ste_s > T1) & (sc_s > T2)) | (ste_s > T1_high)

# Expand speech boundary
def extend_speech(is_speech, h):
    if h <= 0:
        return is_speech.copy()
    x = is_speech.astype(int)
    padded = np.concatenate([np.zeros(h, dtype=int), x, np.zeros(h, dtype=int)])
    csum = np.concatenate([[0], np.cumsum(padded)])
    window_sum = csum[2 * h + 1:] - csum[:-(2 * h + 1)]
    return window_sum > 0

# Shrink speech boundary
def shrink_speech(is_speech, n):
    if n == 0:
        return is_speech.copy()
    if n < 0:
        return extend_speech(is_speech, -n)
    return ~extend_speech(~is_speech, n)

# Remove short silence segments
def fill_short_silences(is_speech, min_silence_frames):
    out = is_speech.copy()
    n, i = len(out), 0
    while i < n:
        if not out[i]:
            j = i
            while j < n and not out[j]:
                j += 1
            if i > 0 and j < n and (j - i) < min_silence_frames:
                out[i:j] = True
            i = j
        else:
            i += 1
    return out

# Convert frame indices to time boundaries
def mask_to_segments(is_speech, fs):
    L = int(round(FRAME_MS * fs / 1000))
    S = int(round(SHIFT_MS * fs / 1000))
    centers = (np.arange(len(is_speech)) * S + L / 2) / fs
    half = S / fs / 2
    segs, start = [], None
    for i, v in enumerate(is_speech):
        if v and start is None:
            start = i
        elif not v and start is not None:
            segs.append([centers[start] - half, centers[i - 1] + half])
            start = None
    if start is not None:
        segs.append([centers[start] - half, centers[-1] + half])
    return segs

# Combine classification and post-processing steps
def segment_signal(feat, T1, T2, T1_high=1.0, hang=0, shrink=0):
    mask = classify_frames(feat["ste_s"], feat["sc_s"], T1, T2, T1_high)
    mask = extend_speech(mask, hang)
    mask = fill_short_silences(mask, MIN_SILENCE_MS // SHIFT_MS)
    mask = shrink_speech(mask, hang + shrink)
    return mask, mask_to_segments(mask, feat["fs"])

# Calculate Error (MAE and RMSE)
def boundary_errors(pred_segs, true_segs):
    pred = np.array([t for sg in pred_segs for t in sg])
    true = np.array([t for sg in true_segs for t in sg])
    if len(true) == 0:
        return np.nan, np.nan
    if len(pred) == 0:
        return np.inf, np.inf
    e1 = [np.min(np.abs(pred - t)) for t in true]
    e2 = [np.min(np.abs(true - p)) for p in pred]
    err = np.array(e1 + e2) * 1000.0
    return float(np.mean(err)), float(np.sqrt(np.mean(err ** 2)))

## 3. Training Module
Determine the best global thresholds by analyzing all training files.

In [22]:
def train(train_dir=TRAIN_DIR):
    files = sorted(glob.glob(os.path.join(train_dir, "*.wav")))
    if not files:
        print(f"Warning: No training files found in '{train_dir}'!")
        return None

    data = []
    for f in files:
        feat = extract_smoothed_features(f)
        feat["true"] = read_lab(f.replace(".wav", ".lab"))
        data.append(feat)

    all_ste = np.concatenate([d["ste_s"] for d in data])
    all_sc = np.concatenate([d["sc_s"] for d in data])
    M1s, M2s, h_s, hs_s, c_s = analyze_histogram(all_ste)
    M1c, M2c, h_c, hs_c, c_c = analyze_histogram(all_sc)

    best = (np.inf, None, None, None, None, None)
    for Ws in W_GRID:
        for Wc in W_GRID:
            T1 = threshold_from_peaks(M1s, M2s, Ws)
            T2 = threshold_from_peaks(M1c, M2c, Wc)
            for Th in T_HIGH_GRID:
                for hang in H_GRID:
                    for shrink in SHRINK_GRID:
                        maes = []
                        for d in data:
                            _, segs = segment_signal(d, T1, T2, Th, hang, shrink)
                            mae, _ = boundary_errors(segs, d["true"])
                            if np.isfinite(mae):
                                maes.append(mae)
                        score = np.mean(maes) if maes else np.inf
                        if score < best[0]:
                            best = (score, Ws, Wc, Th, hang, shrink)

    _, Ws, Wc, Th, hang, shrink = best
    T1 = threshold_from_peaks(M1s, M2s, Ws)
    T2 = threshold_from_peaks(M1c, M2c, Wc)
    
    print(f"\n[TRAINING RESULTS]")
    print(f"Trained on {len(files)} files | Average MAE = {best[0]:.2f} ms")
    print(f"  STE: M1={M1s:.4f} M2={M2s:.4f} W={Ws} => T1={T1:.4f}")
    print(f"  SC : M1={M1c:.4f} M2={M2c:.4f} W={Wc} => T2={T2:.4f}")
    print(f"  High STE: T1_high={Th} | Extension frames: {hang} | Shrink frames: {shrink}")
    
    per_file = [(os.path.basename(f), d["ste_s"], d["sc_s"]) for f, d in zip(files, data)]
    return {"per_file": per_file, "train_mae": float(best[0]), 
            "T1": T1, "T2": T2, "T1_high": Th, "hang": hang, "shrink": shrink, 
            "W_ste": Ws, "W_sc": Wc,
            "ste": (M1s, M2s, h_s, hs_s, c_s), "sc": (M1c, M2c, h_c, hs_c, c_c)}

# Display and save Training Histogram
def save_histogram_figure(model, out_path="histogram_threshold.png"):
    colors = ['tab:red', 'tab:green', 'tab:purple', 'tab:orange', 'tab:brown', 'tab:pink']
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
    for ax, key, idx, T, W, name in [
            (axes[0], "ste", 1, model["T1"], model["W_ste"], "STE"),
            (axes[1], "sc", 2, model["T2"], model["W_sc"], "Spectral Centroid")]:
        M1, M2, hist, smooth, centers = model[key]
        all_x = np.concatenate([p[idx] for p in model["per_file"]])
        lo, hi = all_x.min(), all_x.max()
        width = centers[1] - centers[0]
        
        for j, p in enumerate(model["per_file"]):
            h, c = build_histogram(p[idx], lo=lo, hi=hi)
            ax.plot(c, h / h.sum(), color=colors[j % len(colors)], linewidth=1,
                    alpha=0.6, label=p[0])
            
        ax.plot(centers, smooth / hist.sum(), color='blue', linewidth=2.5,
                label='Smoothed merged histogram')
        ax.axvline(M1, color='green', linestyle=':', linewidth=2.5, label=f'M1 = {M1:.3f}')
        ax.axvline(M2, color='orange', linestyle=':', linewidth=2.5, label=f'M2 = {M2:.3f}')
        ax.axvline(T, color='black', linewidth=3, label=f'Threshold T = {T:.3f} (W = {W})')
        
        ax.bar(centers, hist / hist.sum(), width=width, color='lightgray',
               alpha=0.8, label=f'Merged {len(model["per_file"])} train files')
        
        ax.set_yscale('log')
        ax.set_title(f"Global Threshold for {name}", fontsize=16, fontweight='bold')
        ax.set_xlabel(f"{name} Value (Normalized)", fontsize=14)
        ax.set_ylabel("Frame Ratio (Log Scale)", fontsize=14)
        ax.tick_params(labelsize=12)
        ax.legend(fontsize=10, loc='upper right')
        
    fig.suptitle("Determine Thresholds from Training Data Histogram", fontsize=18, fontweight='bold')
    fig.savefig(out_path, dpi=200)
    plt.show()

## 4. Testing Module
Test the computed global thresholds on the testing data, calculate MAE, and plot comparative figures.

In [23]:
def plot_result(path, feat, pred, true, T1, T2, T1_high, mae, rmse):
    sig, fs = feat["signal"], feat["fs"]
    L = int(round(FRAME_MS * fs / 1000))
    S = int(round(SHIFT_MS * fs / 1000))
    
    t_sig = np.arange(len(sig)) / fs
    t_fr = (np.arange(len(feat["ste"])) * S + L / 2) / fs

    fig, ax = plt.subplots(3, 1, figsize=(10, 8), constrained_layout=True)

    # 1. STE
    ax[0].plot(t_fr, feat["ste"], color='lime', linewidth=1, label='STE (original)')
    ax[0].plot(t_fr, feat["ste_s"], color='cyan', linewidth=1.5, label='STE (filtered)')
    ax[0].axhline(T1, color='black', linewidth=2.5, label=f'T1 = {T1:.3f}')
    if T1_high < 1.0:
        ax[0].axhline(T1_high, color='magenta', linestyle='--', linewidth=1.2,
                      label=f'T1_high = {T1_high:.2f}')
    ax[0].set_xlim(0, t_sig[-1])
    ax[0].set_ylim(0, 1.05)
    ax[0].set_title("Short time energy", fontsize=10, loc="left")
    ax[0].set_xlabel("Time (s)")
    ax[0].set_ylabel("Normalized STE")
    ax[0].legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=4, fontsize=7,
                 frameon=False, columnspacing=1.0, handlelength=1.5)

    # 2. Spectral centroid
    ax[1].plot(t_fr, feat["sc"], color='lime', linewidth=1, label='SC (original)')
    ax[1].plot(t_fr, feat["sc_s"], color='cyan', linewidth=1.5, label='SC (filtered)')
    ax[1].axhline(T2, color='black', linewidth=2.5, label=f'T2 = {T2:.3f}')
    ax[1].set_xlim(0, t_sig[-1])
    ax[1].set_ylim(0, 1.05)
    ax[1].set_title("Spectral centroid", fontsize=10, loc="left")
    ax[1].set_xlabel("Time (s)")
    ax[1].set_ylabel("Normalized SC")
    ax[1].legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=4, fontsize=7,
                 frameon=False, columnspacing=1.0, handlelength=1.5)

    # 3. Waveform + Boundaries
    ax[2].plot(t_sig, sig, color='silver', label='Waveform')
    ax[2].plot(t_fr, feat["ste_s"] * np.max(np.abs(sig)), color='green',
               linewidth=1.5, label='STE overlay')
    for k, (s0, e0) in enumerate(true):
        ax[2].axvline(s0, color='red', linewidth=1.5, label='Groundtruth' if k == 0 else None)
        ax[2].axvline(e0, color='red', linewidth=1.5)
    for k, (s0, e0) in enumerate(pred):
        ax[2].axvline(s0, color='blue', linestyle='--', linewidth=1.5,
                      label='Algorithm' if k == 0 else None)
        ax[2].axvline(e0, color='blue', linestyle='--', linewidth=1.5)
    ax[2].set_xlim(0, t_sig[-1])
    ax[2].set_title("Signal and boundaries", fontsize=10, loc="left")
    ax[2].set_xlabel("Time (s)")
    ax[2].set_ylabel("Amplitude")
    ax[2].legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=4, fontsize=7,
                 frameon=False, columnspacing=1.0, handlelength=1.5)

    fig.suptitle(f"{os.path.basename(path)} | MAE = {mae:.2f} ms | RMSE = {rmse:.2f} ms",
                 fontsize=11, fontweight='bold')
    plt.show()

def test(model, test_dir=TEST_DIR):
    files = sorted(glob.glob(os.path.join(test_dir, "*.wav")))
    if not files:
        print(f"Warning: No testing files found in '{test_dir}'!")
        return
        
    results = []
    for f in files:
        feat = extract_smoothed_features(f)
        _, pred = segment_signal(feat, model["T1"], model["T2"], model["T1_high"], model["hang"],
                                 model.get("shrink", 0))
        true = read_lab(f.replace(".wav", ".lab"))
        mae, rmse = boundary_errors(pred, true)
        
        results.append((os.path.basename(f), len(true) * 2, len(pred) * 2, mae, rmse))
        plot_result(f, feat, pred, true, model["T1"], model["T2"],
                    model["T1_high"], mae, rmse)

    print("\n[TESTING RESULTS]")
    print(f"{'-'*70}")
    print(f"{'File':30s} {'Groundtruth':>12s} {'Algorithm':>10s} {'MAE(ms)':>8s} {'RMSE(ms)':>8s}")
    print(f"{'-'*70}")
    for name, nt, npred, mae, rmse in results:
        print(f"{name:30s} {nt:12d} {npred:10d} {mae:8.2f} {rmse:8.2f}")
    print(f"{'-'*70}")
    ok = [r for r in results if np.isfinite(r[3])]
    if ok:
        print(f"{'AVERAGE':30s} {'':12s} {'':10s} {np.mean([r[3] for r in ok]):8.2f} {np.mean([r[4] for r in ok]):8.2f}")

## 5. Main Execution
Run the training module first, then proceed immediately to testing.

In [24]:
def main():
    print("--- STARTING TRAINING PHASE ---")
    model = train()
    
    if model:
        print("\nGenerating Training Histogram...")
        save_histogram_figure(model)
        
        print("\n--- STARTING TESTING PHASE ---")
        test(model)
    else:
        print("Training failed. Ensure the 'TinHieuHuanLuyen' folder exists and contains valid .wav/.lab files.")

if __name__ == "__main__":
    main()

--- STARTING TRAINING PHASE ---
Training failed. Ensure the 'TinHieuHuanLuyen' folder exists and contains valid .wav/.lab files.
